<h1 align="center"><span style="font-family: Arial, Helvetica, sans-serif; font-size: 22px; font-weight: 700;">WAIGERA'S YOUTUBE VIDEOS</span></h1>

This API call fetches video data from Waigera's YouTube channel and presents it in a clean, organized table.

In [1]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')

CHANNEL_HANDLE = '@Waigera'  

youtube = build('youtube', 'v3', developerKey=API_KEY)

def parse_duration_to_seconds(duration_str):
    """Converts YouTube ISO 8601 duration (e.g., PT14M32S) into integer seconds."""
    seconds = 0
    hours = re.search(r'(\d+)H', duration_str)
    minutes = re.search(r'(\d+)M', duration_str)
    secs = re.search(r'(\d+)S', duration_str)
    
    if hours:
        seconds += int(hours.group(1)) * 3600
    if minutes:
        seconds += int(minutes.group(1)) * 60
    if secs:
        seconds += int(secs.group(1))
    return seconds

def get_waigera_videos_only(handle):
    channel_response = youtube.channels().list(
        part='contentDetails',
        forHandle=handle
    ).execute()
    
    if 'items' not in channel_response or len(channel_response['items']) == 0:
        print(f"❌ Error: Could not find any channel with the handle: {handle}")
        return pd.DataFrame()
        
    uploads_id = channel_response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
    
    video_ids = []
    video_snippets = {}
    next_page_token = None
    
    print(f"Step 1: Gathering all item links from {handle}'s uploads playlist")
    
    while True:
        playlist_response = youtube.playlistItems().list(
            part='snippet,contentDetails',
            playlistId=uploads_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        
        for item in playlist_response.get('items', []):
            v_id = item['contentDetails']['videoId']
            video_ids.append(v_id)
            video_snippets[v_id] = {
                'Title': item['snippet']['title'],
                'Date': item['snippet']['publishedAt'].split('T')[0]
            }
            
        next_page_token = playlist_response.get('nextPageToken')
        if not next_page_token:
            break

    print(f"✅ Found {len(video_ids)} total records. Filtering out Shorts and pulling stats...")
    
    rows = []
    
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        
        video_response = youtube.videos().list(
            part='statistics,contentDetails',
            id=','.join(batch_ids)
        ).execute()
        
        for video_item in video_response.get('items', []):
            v_id = video_item['id']
            
            iso_duration = video_item['contentDetails']['duration']
            duration_in_seconds = parse_duration_to_seconds(iso_duration)
            
            if duration_in_seconds <= 60:
                continue
                
            stats = video_item['statistics']
            rows.append({
                'Date': video_snippets[v_id]['Date'],
                'Title': video_snippets[v_id]['Title'],
                'Views': int(stats.get('viewCount', 0)),
                'Likes': int(stats.get('likeCount', 0)),
                'Comments': int(stats.get('commentCount', 0))
            })
            
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by='Date', ascending=False)
    return df

waigera_table = get_waigera_videos_only(CHANNEL_HANDLE)

if not waigera_table.empty:
    pd.set_option('display.max_colwidth', None)
    print(f"\n📊 {CHANNEL_HANDLE}'s VIDEOS ONLY (Excluding Shorts) - {len(waigera_table)} Videos Found:")
    print(waigera_table.head(20).to_string(index=False))


Step 1: Gathering all item links from @Waigera's uploads playlist
✅ Found 169 total records. Filtering out Shorts and pulling stats...

📊 @Waigera's VIDEOS ONLY (Excluding Shorts) - 167 Videos Found:
      Date                                                             Title  Views  Likes  Comments
2026-10-25                   Inside An 18 MILLION KSH BMW M3 Competition !!!  57068   1963       238
2026-10-19            The 2024 LEXUS RX350: Better Than BMW, AUDI & MERCEDES  76044   1774       208
2026-10-06                     This 500 HP MK7 GTI Is RIDICULOUSLY FAST !!!!  85343   2022       107
2026-09-29            I Spent A Full Day With Kenyan Multi-Millionaires  !!! 168121   2899       104
2026-09-27            SportPesa Racing F1 Car Official unveiling at Concours  25657    626        18
2026-09-25                How A KENYAN BRAND Made It To Formula 1- SportPesa 117596   3464       207
2026-09-20            This Is Why MAINA KAGENI Owns This Car, The BMW X6 !!! 161772   5178   

In [2]:
filtered = waigera_table.to_string()

In [3]:
import os
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')

CHANNEL_HANDLE = '@Waigera'  

youtube = build('youtube', 'v3', developerKey=API_KEY)

def get_all_waigera_uploads(handle):
    channel_response = youtube.channels().list(
        part='contentDetails',
        forHandle=handle
    ).execute()
    
    if 'items' not in channel_response or len(channel_response['items']) == 0:
        print(f"❌ Error: Could not find any channel with the handle: {handle}")
        return pd.DataFrame()
        
    uploads_id = channel_response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
    
    video_ids = []
    video_snippets = {}
    next_page_token = None
    
    print(f"Step 1: Gathering all uploads from {handle}...")
    
    while True:
        playlist_response = youtube.playlistItems().list(
            part='snippet,contentDetails',
            playlistId=uploads_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        
        for item in playlist_response.get('items', []):
            v_id = item['contentDetails']['videoId']
            video_ids.append(v_id)
            video_snippets[v_id] = {
                'Title': item['snippet']['title'],
                'Date': item['snippet']['publishedAt'].split('T')[0] # Clean YYYY-MM-DD
            }
            
        next_page_token = playlist_response.get('nextPageToken')
        if not next_page_token:
            break

    print(f"Found {len(video_ids)} total uploads. Gathering analytics metrics...")
    
    rows = []
    
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        
        video_response = youtube.videos().list(
            part='statistics',
            id=','.join(batch_ids)
        ).execute()
        
        for video_item in video_response.get('items', []):
            v_id = video_item['id']
            stats = video_item['statistics']
            
            rows.append({
                'Date': video_snippets[v_id]['Date'],
                'Title': video_snippets[v_id]['Title'],
                'Views': int(stats.get('viewCount', 0)),
                'Likes': int(stats.get('likeCount', 0)),
                'Comments': int(stats.get('commentCount', 0))
            })
            
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(by='Date', ascending=False)
    return df

waigera_table = get_all_waigera_uploads(CHANNEL_HANDLE)

if not waigera_table.empty:
    pd.set_option('display.max_colwidth', None)
    print(f"\n{CHANNEL_HANDLE}'s TOTAL UPLOADS - {len(waigera_table)} Items Found:")
    print(waigera_table.head(20).to_string(index=False))


Step 1: Gathering all uploads from @Waigera...
Found 169 total uploads. Gathering analytics metrics...

@Waigera's TOTAL UPLOADS - 169 Items Found:
      Date                                                             Title  Views  Likes  Comments
2026-10-25                   Inside An 18 MILLION KSH BMW M3 Competition !!!  57068   1963       238
2026-10-19            The 2024 LEXUS RX350: Better Than BMW, AUDI & MERCEDES  76044   1774       208
2026-10-06                     This 500 HP MK7 GTI Is RIDICULOUSLY FAST !!!!  85343   2022       107
2026-09-29            I Spent A Full Day With Kenyan Multi-Millionaires  !!! 168121   2899       104
2026-09-27            SportPesa Racing F1 Car Official unveiling at Concours  25657    626        18
2026-09-25                How A KENYAN BRAND Made It To Formula 1- SportPesa 117596   3464       207
2026-09-20            This Is Why MAINA KAGENI Owns This Car, The BMW X6 !!! 161772   5178       149
2026-09-14             Inside A 33 MILLION K

In [4]:
full_data = (waigera_table.to_string())

In [5]:
def parse_video_table(table_text):
    rows = []
    pattern = r"^\s*\d+\s+(\d{4}-\d{2}-\d{2})\s+(.*?)\s+(\d+)\s+(\d+)\s+(\d+)\s*$"

    for line in table_text.splitlines():
        match = re.match(pattern, line)
        if match:
            date, title, views, likes, comments = match.groups()
            rows.append({
                "Date": date,
                "Title": title.strip(),
                "Views": int(views),
                "Likes": int(likes),
                "Comments": int(comments),
            })

    return pd.DataFrame(rows)

full_df = parse_video_table(full_data)
filtered_df = parse_video_table(filtered)

filtered_keys = set(zip(filtered_df["Date"], filtered_df["Title"]))
only_in_full = full_df[
    ~full_df.apply(lambda row: (row["Date"], row["Title"]) in filtered_keys, axis=1)
]

only_in_full

,Date,Title,Views,Likes,Comments
34,2026-02-27,Waigera Live Stream,0,0,0
124,2025-01-24,The NEWEST!! Porsche 911 DAKAR in Kenya 🔥#porsche #kenya #porsche911 #viralvideo #shorts #cars,66943,1474,15


In [6]:
final = full_df.drop(index=[34, 124, 65, 20, 21]).reset_index(drop=True)
print(final.to_string())


           Date                                                                                   Title   Views  Likes  Comments
0    2026-10-25                                         Inside An 18 MILLION KSH BMW M3 Competition !!!   57068   1963       238
1    2026-10-19                                  The 2024 LEXUS RX350: Better Than BMW, AUDI & MERCEDES   76044   1774       208
2    2026-10-06                                           This 500 HP MK7 GTI Is RIDICULOUSLY FAST !!!!   85343   2022       107
3    2026-09-29                                  I Spent A Full Day With Kenyan Multi-Millionaires  !!!  168121   2899       104
4    2026-09-27                                  SportPesa Racing F1 Car Official unveiling at Concours   25657    626        18
5    2026-09-25                                      How A KENYAN BRAND Made It To Formula 1- SportPesa  117596   3464       207
6    2026-09-20                                  This Is Why MAINA KAGENI Owns This Car, The BMW 